In [7]:
# =========================================================
# Step A1: Load Dataset
# =========================================================

import pandas as pd
import numpy as np

# Load dataset (download abalone.csv from UCI)
columns = [
    "Sex",
    "Length",
    "Diameter",
    "Height",
    "WholeWeight",
    "ShuckedWeight",
    "VisceraWeight",
    "ShellWeight",
    "Rings"
]

data = pd.read_csv("abalone.data", names=columns)

print("First 5 rows:\n")
print(data.head())

print("\nNumber of rows:", len(data))
print("\nColumns:", data.columns)


# Checkpoint:
# input: physical measurements of abalone (length, diameter, height, etc.)
# output: age of abalone
# why output is numeric: because age is a measurable quantity (continuous value)


# =========================================================
# Step A2: Convert Target
# =========================================================

# According to dataset description:
# Age = Rings + 1.5

data["y"] = data["Rings"] + 1.5


# =========================================================
# Step A3: Choose Features
# =========================================================

# Select 3 numeric features

features = ["Length", "Diameter", "Height"]

X = data[features].values
y = data["y"].values.reshape(-1,1)

# Justification:
# Feature 1: Length → larger abalone likely older
# Feature 2: Diameter → shell size grows with age
# Feature 3: Height → thickness may correlate with maturity


# =========================================================
# Step A4: Train Test Split (80/20 manually)
# =========================================================

N = len(X)
split_index = int(0.8 * N)

X_train = X[:split_index]
X_test = X[split_index:]

y_train = y[:split_index]
y_test = y[split_index:]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)


# =========================================================
# Step A5: Normalize Inputs
# =========================================================

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# Checkpoint:
# normalization prevents features with large values dominating learning
# it also stabilizes gradient descent


# =========================================================
# Part B: Forward Model
# =========================================================

def forward(X, w, b):
    """
    compute prediction y_hat = Xw + b
    """
    y_hat = X.dot(w) + b

    print("Shape of X:", X.shape)
    print("Shape of w:", w.shape)
    print("Shape of b:", np.shape(b))
    print("Shape of y_hat:", y_hat.shape)

    return y_hat


# =========================================================
# Part C: Mean Squared Error
# =========================================================

def mse(y, y_hat):
    loss = np.mean((y - y_hat) ** 2)
    return loss

# Checkpoint:
# why square: prevents positive and negative errors cancelling out
# large errors become very expensive


# =========================================================
# Part D: Gradients
# =========================================================

def grad_w(X, y, y_hat):
    N = len(y)
    dW = (2/N) * X.T.dot(y_hat - y)
    return dW

def grad_b(y, y_hat):
    N = len(y)
    db = (2/N) * np.sum(y_hat - y)
    return db

# Checkpoint:
# gradient means slope of the loss function
# it tells direction where loss increases
# subtracting gradient moves parameters toward lower loss


# =========================================================
# Part E: Training Loop
# =========================================================

# initialize parameters

d = X_train.shape[1]
w = np.random.randn(d,1) * 0.01
b = 0.0

lr = 0.01
epochs = 500

for epoch in range(epochs):

    # forward
    y_hat = X_train.dot(w) + b

    # loss
    loss = mse(y_train, y_hat)

    # gradients
    dW = grad_w(X_train, y_train, y_hat)
    db = grad_b(y_train, y_hat)

    # update
    w = w - lr * dW
    b = b - lr * db

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss)

# Initial expectation:
# loss should decrease gradually

# Revised expectation:
# loss decreases steadily but not instantly


# =========================================================
# Part F: Evaluation
# =========================================================

# predictions on test set
y_pred = X_test.dot(w) + b

# Test MSE
test_mse = np.mean((y_test - y_pred)**2)

# Test MAE
test_mae = np.mean(np.abs(y_test - y_pred))

print("\nTest MSE:", test_mse)
print("Test MAE:", test_mae)


# =========================================================
# Print 5 example predictions
# =========================================================

print("\nExample Predictions:\n")

for i in range(5):
    true_age = y_test[i][0]
    predicted_age = y_pred[i][0]
    error = abs(true_age - predicted_age)

    print("True Age:", true_age,
          "| Predicted Age:", predicted_age,
          "| Absolute Error:", error)


First 5 rows:

  Sex  Length  Diameter  Height  WholeWeight  ShuckedWeight  VisceraWeight  \
0   M   0.455     0.365   0.095       0.5140         0.2245         0.1010   
1   M   0.350     0.265   0.090       0.2255         0.0995         0.0485   
2   F   0.530     0.420   0.135       0.6770         0.2565         0.1415   
3   M   0.440     0.365   0.125       0.5160         0.2155         0.1140   
4   I   0.330     0.255   0.080       0.2050         0.0895         0.0395   

   ShellWeight  Rings  
0        0.150     15  
1        0.070      7  
2        0.210      9  
3        0.155     10  
4        0.055      7  

Number of rows: 4177

Columns: Index(['Sex', 'Length', 'Diameter', 'Height', 'WholeWeight', 'ShuckedWeight',
       'VisceraWeight', 'ShellWeight', 'Rings'],
      dtype='object')
Train shape: (3341, 3) (3341, 1)
Test shape: (836, 3) (836, 1)
Epoch: 0 Loss: 144.35095469016974
Epoch: 50 Loss: 24.939381853388916
Epoch: 100 Loss: 9.599655230238252
Epoch: 150 Loss: 7.56256